In [1]:
import pandas as pd
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    TargetEncoder,
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    MaxAbsScaler,
)

In [2]:
df = pd.read_csv("houses_raw.csv")
print(df.head())
print(df.shape)
print(df.dtypes)

   area  bedrooms  bathrooms     finishing   floor    price
0    86         2          2  Not Finished  Fourth  2061000
1   210         3          2      Finished   First  4230000
2   188         4          4      Finished   First  3995000
3   149         1          1  Not Finished  Fourth  2824000
4   148         2          1      Finished   Third  3079000
(150, 6)
area         int64
bedrooms     int64
bathrooms    int64
finishing      str
floor          str
price        int64
dtype: object


In [3]:
le= LabelEncoder()
label_df= df.copy()
label_df["finishing"] = le.fit_transform(df["finishing"])

print("finishing classes: ", list(le.classes_))

finishing classes:  ['Finished', 'Not Finished']


In [4]:
con = pd.concat([label_df["finishing"], df["finishing"]], axis=1)
print(con.head())

   finishing     finishing
0          1  Not Finished
1          0      Finished
2          0      Finished
3          1  Not Finished
4          0      Finished


In [5]:
label_df["floor"]= le.fit_transform(df["floor"])
con2=pd.concat([df["floor"], label_df["floor"]], axis=1)
print(con2.head())

    floor  floor
0  Fourth      1
1   First      0
2   First      0
3  Fourth      1
4   Third      3


In [6]:
ohe= OneHotEncoder(sparse_output=False)
ohe_array= ohe.fit_transform(df[["finishing", "floor"]])
ohe_cols= ohe.get_feature_names_out(["finishing", "floor"])

dff= pd.DataFrame(ohe_array, columns=ohe_cols)

print(dff)

     finishing_Finished  finishing_Not Finished  floor_First  floor_Fourth  \
0                   0.0                     1.0          0.0           1.0   
1                   1.0                     0.0          1.0           0.0   
2                   1.0                     0.0          1.0           0.0   
3                   0.0                     1.0          0.0           1.0   
4                   1.0                     0.0          0.0           0.0   
..                  ...                     ...          ...           ...   
145                 0.0                     1.0          0.0           0.0   
146                 1.0                     0.0          0.0           0.0   
147                 1.0                     0.0          1.0           0.0   
148                 1.0                     0.0          1.0           0.0   
149                 1.0                     0.0          0.0           0.0   

     floor_Second  floor_Third  
0             0.0          0.0

In [7]:
te= TargetEncoder(target_type="continuous")
target_df= df.copy()
target_df[["finishig", "floor"]]= te.fit_transform(df[["finishing", "floor"]], df["price"])

print(target_df.head())

   area  bedrooms  bathrooms     finishing         floor    price  \
0    86         2          2  Not Finished  3.659673e+06  2061000   
1   210         3          2      Finished  3.570023e+06  4230000   
2   188         4          4      Finished  3.437774e+06  3995000   
3   149         1          1  Not Finished  3.659673e+06  2824000   
4   148         2          1      Finished  3.211950e+06  3079000   

       finishig  
0  3.068449e+06  
1  3.815565e+06  
2  3.739039e+06  
3  3.068449e+06  
4  3.815565e+06  


In [8]:
x= label_df.drop(columns=["price"])
y= label_df["price"]

In [9]:
scalers={
    "MinMax": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Robust": RobustScaler(),
    "MaxAbs": MaxAbsScaler()
}

for name, scaler in scalers.items():
    scaled = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)
    print(f"\n{name}")
    print("-"*60)
    print (scaled.head().round(3))


MinMax
------------------------------------------------------------
    area  bedrooms  bathrooms  finishing  floor
0  0.084      0.25      0.333        1.0  0.333
1  0.781      0.50      0.333        0.0  0.000
2  0.657      0.75      1.000        0.0  0.000
3  0.438      0.00      0.000        1.0  0.333
4  0.433      0.25      0.000        0.0  1.000

Standard
------------------------------------------------------------
    area  bedrooms  bathrooms  finishing  floor
0 -1.547    -0.634     -0.302      1.259 -0.303
1  0.952     0.215     -0.302     -0.794 -1.160
2  0.509     1.063      1.476     -0.794 -1.160
3 -0.277    -1.482     -1.192      1.259 -0.303
4 -0.297    -0.634     -1.192     -0.794  1.411

Robust
------------------------------------------------------------
    area  bedrooms  bathrooms  finishing  floor
0 -0.923      -0.5        0.0        1.0  0.000
1  0.499       0.0        0.0        0.0 -0.364
2  0.246       0.5        1.0        0.0 -0.364
3 -0.201      -1.0     

In [ ]:
x #Scaling is for the features only (x), not the target variable (y).

,area,bedrooms,bathrooms,finishing,floor
0,86,2,2,1,1
1,210,3,2,0,0
2,188,4,4,0,0
3,149,1,1,1,1
4,148,2,1,0,3
...,...,...,...,...,...
145,224,4,3,1,3
146,76,2,2,0,3
147,112,2,1,0,0
148,218,3,2,0,0


In [ ]:
y # the target remainst the same unscaled. It is not recommended to scale the target variable in regression problems, as it can affect the interpretation of the model's coefficients and predictions.

0      2061000
1      4230000
2      3995000
3      2824000
4      3079000
        ...   
145    4610000
146    2377000
147    2492000
148    4425000
149    1639000
Name: price, Length: 150, dtype: int64

In [13]:
final_scaler= StandardScaler()
x_scaled= pd.DataFrame(final_scaler.fit_transform(x), columns=x.columns)

clean_df= x_scaled.copy()
clean_df["price"]=y

clean_df.to_csv("houses_clean.csv", index=False)

print(clean_df.head())
print(clean_df.shape)

       area  bedrooms  bathrooms  finishing     floor    price
0 -1.546882 -0.633507  -0.302369   1.259447 -0.302778  2061000
1  0.951989  0.214940  -0.302369  -0.793999 -1.159697  4230000
2  0.508641  1.063387   1.476270  -0.793999 -1.159697  3995000
3 -0.277294 -1.481954  -1.191688   1.259447 -0.302778  2824000
4 -0.297446 -0.633507  -1.191688  -0.793999  1.411060  3079000
(150, 6)
